In [22]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

In [2]:
print("Current Working Directory:", os.getcwd())
print("Files in this folder:", os.listdir('.'))

Current Working Directory: d:\Visu\JNTU\Projects\VigilantFlow\src\data
Files in this folder: ['make_dataset.ipynb']


In [3]:
df = pd.read_csv("../../data/raw/sensor.csv")


In [4]:
df.head()

,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,...,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,...,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,...,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,...,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL
3,3,2018-04-01 00:03:00,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,...,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,240.4514,203.1250,NORMAL
4,4,2018-04-01 00:04:00,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,...,41.40625,38.773150,65.10416,51.79398,38.773150,158.2755,66.55093,242.1875,201.3889,NORMAL


In [5]:
df.shape

(220320, 55)

In [6]:
df.isnull().sum()

Unnamed: 0             0
timestamp              0
sensor_00          10208
sensor_01            369
sensor_02             19
sensor_03             19
sensor_04             19
sensor_05             19
sensor_06           4798
sensor_07           5451
sensor_08           5107
sensor_09           4595
sensor_10             19
sensor_11             19
sensor_12             19
sensor_13             19
sensor_14             21
sensor_15         220320
sensor_16             31
sensor_17             46
sensor_18             46
sensor_19             16
sensor_20             16
sensor_21             16
sensor_22             41
sensor_23             16
sensor_24             16
sensor_25             36
sensor_26             20
sensor_27             16
sensor_28             16
sensor_29             72
sensor_30            261
sensor_31             16
sensor_32             68
sensor_33             16
sensor_34             16
sensor_35             16
sensor_36             16
sensor_37             16


1. Here the column "sensor_15" is completely filled with null values and "sensor_50" has 30% null values: drop them
2. We are dealing with time-series data, so instead of dropping all the null value rows, we interpolate them.
3. Drop "Unnamed" and "timestamp"

In [7]:
df.drop(columns=["Unnamed: 0", "timestamp", "sensor_15"], inplace=True)

In [8]:
df.drop(columns=["sensor_50"], inplace=True)

In [9]:
df.head()

,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,...,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,machine_status
0,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,...,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,NORMAL
1,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,...,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,NORMAL
2,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,...,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,203.7037,NORMAL
3,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,15.69734,15.08247,...,31.510420,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,203.1250,NORMAL
4,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,15.69734,15.08247,...,31.510420,41.40625,38.773150,65.10416,51.79398,38.773150,158.2755,66.55093,201.3889,NORMAL


In [10]:
status = df["machine_status"]
features = df.drop(columns=["machine_status"])

In [11]:
features = features.interpolate(method='linear')

In [12]:
features = features.bfill()

In [13]:
status.unique()

array(['NORMAL', 'BROKEN', 'RECOVERING'], dtype=object)

Here we filter the rows that have "NORMAL" status. This is to train the LSTM on healthy data so that it does not consider other situations and see them as normal condition for the pump.

In [14]:
features['machine_status'] = status
normal_data = features[features['machine_status'] == 'NORMAL'].drop(columns=['machine_status'])

Here we need to scale our columns as LSTMs are highly sensitive to scale of the input

In [23]:
scaler = MinMaxScaler()
scaled_normal_data = scaler.fit_transform(normal_data)

In [26]:
scaled_normal_data

array([[0.9671944 , 0.83014531, 0.84848441, ..., 0.2422596 , 0.08962265,
        0.17587341],
       [0.9671944 , 0.83014531, 0.84848441, ..., 0.2422596 , 0.08962265,
        0.17587341],
       [0.95908931, 0.83473604, 0.84848441, ..., 0.23845725, 0.08827493,
        0.17826216],
       ...,
       [0.9401777 , 0.84085688, 0.70396245, ..., 0.35361206, 0.29380043,
        0.20752465],
       [0.94403723, 0.84085688, 0.70396256, ..., 0.36338948, 0.29043129,
        0.20961476],
       [0.9401777 , 0.84085688, 0.70396256, ..., 0.37262362, 0.28234509,
        0.20961476]], shape=(205836, 50))

In [27]:
# Saving the scaled data for usage in test data
os.makedirs('../../models', exist_ok=True)
joblib.dump(scaler, '../../models/scaler.pkl')

['../../models/scaler.pkl']

In [29]:
os.makedirs('../../data/processed', exist_ok=True)
np.save('../../data/processed/training_data.npy', scaled_normal_data)

Track the .npy and .pkl file using dvc and push changes to github